# Tests for `newrelic_sb_sdk.shortcuts.nrql`

## Imports

In [ ]:
import warnings
from unittest.mock import MagicMock, patch

import pytest

In [ ]:
from newrelic_sb_sdk.client import NewRelicClient
from newrelic_sb_sdk.graphql.objects import Account
from newrelic_sb_sdk.shortcuts.nrql import (
    nrql,
    perform_nrql_query,
)
from newrelic_sb_sdk.utils.exceptions import NewRelicError

## Tests

In [ ]:
class TestShortcutsNrql:
    def test_nrql(self):
        query = """
            SELECT *
            FROM Transaction
        """
        result = nrql(query)
        assert result == "SELECT *\n            FROM Transaction"

    @patch("newrelic_sb_sdk.shortcuts.nrql.NewRelicClient.execute")
    def test_perform_nrql_query_completed_immediately(self, mock_execute):
        mock_response = MagicMock()
        mock_response.json.return_value = {
            "data": {
                "actor": {
                    "nrql": {
                        "results": [{"count": 123}],
                        "queryProgress": {
                            "completed": True,
                            "queryId": "some_query_id",
                            "retryAfter": 0,
                        },
                    }
                }
            }
        }
        mock_execute.return_value = mock_response

        client = NewRelicClient(new_relic_user_key="NRAK-123456789012345678901234567")
        account = Account(json_data={"id": 123, "name": "Acme Corp"})

        results = perform_nrql_query(
            client=client,
            account=account,
            nrql_query=nrql("SELECT count(*) FROM Transaction"),
        )

        assert len(results) == 1
        assert results[0].get("count") == 123

    @patch("time.sleep", return_value=None)
    @patch("newrelic_sb_sdk.shortcuts.nrql.NewRelicClient.execute")
    def test_perform_nrql_query_completed_after_retry(self, mock_execute, mock_sleep):
        resp_1 = MagicMock()
        resp_1.json.return_value = {
            "data": {
                "actor": {
                    "nrql": {
                        "results": [],
                        "queryProgress": {
                            "completed": False,
                            "queryId": "some_query_id",
                            "retryAfter": 1,
                        },
                    }
                }
            }
        }

        resp_2 = MagicMock()
        resp_2.json.return_value = {
            "data": {
                "actor": {
                    "nrqlQueryProgress": {
                        "results": [{"count": 123}],
                        "queryProgress": {
                            "completed": True,
                            "queryId": "some_query_id",
                            "retryAfter": 0,
                        },
                    }
                }
            }
        }

        mock_execute.side_effect = [resp_1, resp_2]

        client = NewRelicClient(new_relic_user_key="NRAK-123456789012345678901234567")
        account = Account(json_data={"id": 123, "name": "Acme Corp"})

        results = perform_nrql_query(
            client=client,
            account=account,
            nrql_query=nrql("SELECT count(*) FROM Transaction"),
        )

        assert len(results) == 1
        assert results[0].get("count") == 123
        assert mock_execute.call_count == 2
        assert mock_sleep.call_count == 1

    @patch("time.sleep", return_value=None)
    @patch("newrelic_sb_sdk.shortcuts.nrql.NewRelicClient.execute")
    def test_perform_nrql_query_retry_and_fail(self, mock_execute, mock_sleep):
        mock_execute.side_effect = NewRelicError()

        client = NewRelicClient(new_relic_user_key="NRAK-123456789012345678901234567")
        account = Account(json_data={"id": 123, "name": "Acme Corp"})

        with pytest.raises(NewRelicError):
            perform_nrql_query(
                client=client,
                account=account,
                nrql_query=nrql("SELECT count(*) FROM Transaction"),
                max_retries=3,
                retry_delay=1,
            )

        assert mock_execute.call_count == 3
        assert mock_sleep.call_count == 2

    @patch("time.sleep", return_value=None)
    @patch("newrelic_sb_sdk.shortcuts.nrql._perform_nrql_query")
    def test_perform_nrql_query_deprecated_max_retry_warning(
        self, mock_perform, mock_sleep
    ):
        client = NewRelicClient(new_relic_user_key="NRAK-123456789012345678901234567")
        account = Account(json_data={"id": 123, "name": "Acme Corp"})

        mock_perform.side_effect = NewRelicError()

        with pytest.warns(DeprecationWarning):
            with pytest.raises(NewRelicError):
                perform_nrql_query(
                    client=client,
                    account=account,
                    nrql_query=nrql("SELECT count(*) FROM Transaction"),
                    max_retry=2,
                    retry_delay=1,
                )

        assert mock_perform.call_count == 2